# 🏐 LNV Scraper - Expérimentation & Amélioration

Ce notebook permet d'expérimenter et d'améliorer l'identification des matchs LNV (compétitions professionnelles) depuis le site FFVB.

**Problème résolu :**
- Les poules LNV (AALNV) étaient identifiées mais retournaient **0 feuille de match**
- Les PDFs sont hébergés sur `lnv.fr`, pas sur `ffvbbeach.org` via `ffvolley_fdme.php`
- Le scraper ne cherchait que les formulaires `ffvolley_fdme.php`, ignorant les liens `lnv.fr`

**Solution :**
- Détection des formulaires pointant vers `lnv.fr/pdf/...` en plus de `ffvolley_fdme.php`
- Extraction du code match depuis les URLs LNV (`/MSL001-2026.pdf`)
- Support des URL directes pour le téléchargement

In [1]:
# Imports
import sys, os, re, json
from pathlib import Path
from collections import Counter
from urllib.parse import urljoin, urlencode

import pandas as pd
from bs4 import BeautifulSoup

# Ajouter le projet au path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root / "src"))

from pyvolley.scrapers.ffvb import FFVBScraper, PouleInfo
from pyvolley.scrapers.lnv import LNVScraper, PRO_COMPETITIONS, PRO_ENTITY_CODE
from pyvolley.scrapers.base import MatchInfo

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_rows", 50)

scraper = FFVBScraper()
lnv = LNVScraper(scraper)
print("✅ Modules chargés")

✅ Modules chargés


## 1. Explorer la structure des pages LNV

Commençons par examiner la structure HTML d'une page de calendrier AALNV pour comprendre comment les matchs y sont référencés.

In [2]:
# Récupérer les poules AALNV pour la saison en cours
saison = "2025/2026"
poules = scraper.get_poules_for_entity("AALNV", saison)

df_poules = pd.DataFrame([
    {"Code": p.code, "Nom": p.nom, "Entité": p.entity_code, "Saison": p.saison}
    for p in poules
])
print(f"📋 {len(poules)} poules LNV découvertes pour {saison}")
df_poules

📋 12 poules LNV découvertes pour 2025/2026


,Code,Nom,Entité,Saison
0,FAZ,FAZ SAFORELLE POWER 6 - PLAYOFFS,AALNV,2025/2026
1,SPS,SPS SAFORELLE POWER 6,AALNV,2025/2026
2,MSL,MSL MARMARA SPIKELIGUE,AALNV,2025/2026
3,PAZ,PAZ MARMARA SPIKELIGUE - PLAYOFFS,AALNV,2025/2026
4,DAZ,DAZ LIGUE A MASCULINE - QUALIFICATION EUROPE,AALNV,2025/2026
5,LBM,LBM LIGUE B MASCULINE,AALNV,2025/2026
6,PBA,PBA LIGUE B MASCULINE - PLAYOFFS - POULE A,AALNV,2025/2026
7,PBB,PBB LIGUE B MASCULINE - PLAYOFFS - POULE B,AALNV,2025/2026
8,PBZ,PBZ LIGUE B MASCULINE - PLAYOFFS,AALNV,2025/2026
9,TSA,TSA TEST POULE FDME A,AALNV,2025/2026


In [ ]:
# Analyser la structure HTML d'une page de calendrier LNV (MSL = Marmara SpikeLigue)
params = {
    "saison": saison,
    "codent": "AALNV",
    "poule": "MSL",
    "calend": "COMPLET"
}
url = urljoin(scraper.base_url, f"vbspo_calendrier.php?{urlencode(params)}")
soup = scraper.client.get_soup(url)

# Analyser les types de formulaires
actions = Counter()
for f in soup.find_all("form"):
    action = f.get("action", "")
    if "lnv.fr" in action:
        actions["lnv.fr PDF"] += 1
    elif "fdme" in action:
        actions["ffvolley_fdme.php"] += 1
    elif "calendrier" in action:
        actions["calendrier"] += 1
    elif "planning" in action:
        actions["planning"] += 1
    else:
        actions["autre"] += 1

print(f"📊 Formulaires sur la page calendrier MSL ({url[:60]}...):")
for label, count in actions.most_common():
    marker = "✅" if label == "lnv.fr PDF" else "📋"
    print(f"  {marker} {label}: {count}")

print(f"\n→ Les matchs LNV sont uniquement via lnv.fr, PAS ffvolley_fdme.php")

AttributeError: 'FFVBScraper' object has no attribute '_get_soup'

## 2. Analyser les URLs de match LNV

Examinons les URLs des PDFs LNV pour comprendre le pattern et extraire les codes match.

In [4]:
# Extraire toutes les URLs LNV de la page MSL
lnv_urls = []
for f in soup.find_all("form"):
    action = f.get("action", "")
    if "lnv.fr" in action and ".pdf" in action:
        # Extraire le contexte de la ligne du tableau
        tr = f.find_parent("tr")
        row_data = {}
        if tr:
            cells = [c.get_text(strip=True) for c in tr.find_all("td")]
            row_data["cells"] = cells
        
        code_match = re.search(r"/([A-Z0-9]{3,6}\d{3})-(\d{4})\.pdf", action)
        lnv_urls.append({
            "url": action,
            "code": code_match.group(1) if code_match else "?",
            "year": code_match.group(2) if code_match else "?",
            "row": row_data.get("cells", []),
        })

# Afficher les premiers
df_urls = pd.DataFrame(lnv_urls)
print(f"🔗 {len(lnv_urls)} URLs de PDFs LNV trouvées\n")
print("Pattern URL: https://www.lnv.fr/pdf/{year}/DataVolley/{gender}/{code}-{year}.pdf")
print()
df_urls[["code", "url"]].head(10)

NameError: name 'soup' is not defined

In [ ]:
# Inspecter la structure d'une ligne de match LNV
sample = lnv_urls[0]
print(f"Code match: {sample['code']}")
print(f"URL PDF:    {sample['url']}")
print(f"\nCellules de la ligne:")
for i, cell in enumerate(sample["row"]):
    print(f"  [{i}] {cell}")

## 3. Tester la détection de matchs (avant/après correction)

Vérifions que la correction apportée à `get_matches_for_poule` fonctionne sur toutes les poules LNV.

In [5]:
# Compter les matchs par poule via le scraper corrigé
results = []
for p in poules:
    matches = list(scraper.get_matches_for_poule("AALNV", p.code, saison))
    results.append({
        "Poule": p.code,
        "Nom": p.nom,
        "Matchs détectés": len(matches),
        "Avant correction": 0,  # Le code précédent retournait toujours 0
    })

df_results = pd.DataFrame(results)
total = df_results["Matchs détectés"].sum()
print(f"🏆 Total matchs LNV détectés: {total} (avant: 0)")
print()
df_results

🏆 Total matchs LNV détectés: 369 (avant: 0)



,Poule,Nom,Matchs détectés,Avant correction
0,FAZ,FAZ SAFORELLE POWER 6 - PLAYOFFS,0,0
1,SPS,SPS SAFORELLE POWER 6,124,0
2,MSL,MSL MARMARA SPIKELIGUE,147,0
3,PAZ,PAZ MARMARA SPIKELIGUE - PLAYOFFS,0,0
4,DAZ,DAZ LIGUE A MASCULINE - QUALIFICATION EUROPE,0,0
5,LBM,LBM LIGUE B MASCULINE,98,0
6,PBA,PBA LIGUE B MASCULINE - PLAYOFFS - POULE A,0,0
7,PBB,PBB LIGUE B MASCULINE - PLAYOFFS - POULE B,0,0
8,PBZ,PBZ LIGUE B MASCULINE - PLAYOFFS,0,0
9,TSA,TSA TEST POULE FDME A,0,0


## 4. Vérifier les données extractibles des lignes de match

Chaque ligne du calendrier contient des informations précieuses : code match, date, heure, équipes, scores, détails des sets.

In [6]:
# Extraire les informations détaillées de chaque ligne de match
match_data = []
for entry in lnv_urls:
    cells = entry["row"]
    if len(cells) >= 6:
        match_data.append({
            "Code": entry["code"],
            "Date": cells[1] if len(cells) > 1 else "",
            "Heure": cells[2] if len(cells) > 2 else "",
            "Équipe A": cells[3] if len(cells) > 3 else "",
            "Équipe B": cells[5] if len(cells) > 5 else "",
            "Sets A": cells[6] if len(cells) > 6 else "",
            "Sets B": cells[7] if len(cells) > 7 else "",
            "Détails sets": cells[8] if len(cells) > 8 else "",
            "Score": cells[9] if len(cells) > 9 else "",
        })

df_matches = pd.DataFrame(match_data)
print(f"📊 {len(match_data)} matchs avec données détaillées")
df_matches.head(10)

📊 0 matchs avec données détaillées


""


## 5. Comparer avec la détection FFVB classique (ABCCS)

Vérifions que les entités non-LNV continuent de fonctionner correctement via `ffvolley_fdme.php`.

In [7]:
# Vérifier que le scraper FFVB classique fonctionne toujours
comparison = []
for entity_code, poule_code in [("ABCCS", "EFA"), ("ABCCS", "EMA"), ("LIRA", "PMA")]:
    try:
        matches = list(scraper.get_matches_for_poule(entity_code, poule_code, saison))
        if matches:
            sample = matches[0]
            is_lnv = "lnv.fr" in (sample.pdf_url or "")
        else:
            is_lnv = False
        comparison.append({
            "Entité": entity_code,
            "Poule": poule_code,
            "Matchs": len(matches),
            "Source PDF": "lnv.fr" if is_lnv else "ffvbbeach.org",
            "Exemple URL": (matches[0].pdf_url[:60] + "...") if matches else "-",
        })
    except Exception as e:
        comparison.append({
            "Entité": entity_code, "Poule": poule_code,
            "Matchs": 0, "Source PDF": f"erreur: {e}", "Exemple URL": "-"
        })

# Ajouter les résultats LNV
for code in ["MSL", "SPS", "LBM"]:
    ms = list(scraper.get_matches_for_poule("AALNV", code, saison))
    comparison.append({
        "Entité": "AALNV",
        "Poule": code,
        "Matchs": len(ms),
        "Source PDF": "lnv.fr" if ms and "lnv.fr" in (ms[0].pdf_url or "") else "ffvbbeach.org",
        "Exemple URL": (ms[0].pdf_url[:60] + "...") if ms else "-",
    })

df_cmp = pd.DataFrame(comparison)
print("📊 Comparaison détection FFVB classique vs LNV")
df_cmp

📊 Comparaison détection FFVB classique vs LNV


,Entité,Poule,Matchs,Source PDF,Exemple URL
0,ABCCS,EFA,74,ffvbbeach.org,https://www.ffvbbeach.org/ffvbapp/resu/ffvolley_fdme.php?sai...
1,ABCCS,EMA,111,ffvbbeach.org,https://www.ffvbbeach.org/ffvbapp/resu/ffvolley_fdme.php?sai...
2,LIRA,PMA,78,ffvbbeach.org,https://www.ffvbbeach.org/ffvbapp/resu/ffvolley_fdme.php?sai...
3,AALNV,MSL,147,lnv.fr,https://www.lnv.fr/pdf/2025/DataVolley/Men/MSL001-2026.pdf...
4,AALNV,SPS,124,lnv.fr,https://www.lnv.fr/pdf/2025/DataVolley/Women/SPS001-2026.pdf...
5,AALNV,LBM,98,lnv.fr,https://www.lnv.fr/pdf/2025/DataVolley/Men/LBM001-2026.pdf...


## 6. Vue d'ensemble du scraper LNV corrigé

Vérification complète via le `LNVScraper` qui encapsule la logique.

In [8]:
# Découverte dynamique des compétitions + comptage des matchs
discovered = lnv.discover_competitions(saison)
counts = lnv.count_matches(saison)

summary = []
for p in discovered:
    count = counts.get(p.code, -1)
    known = any(c.code == p.code for c in PRO_COMPETITIONS)
    summary.append({
        "Code": p.code,
        "Nom": p.nom,
        "Matchs": count if count >= 0 else "erreur",
        "Connue": "✅" if known else "🆕",
    })

df_summary = pd.DataFrame(summary)
total = sum(c for c in counts.values() if c > 0)
print(f"🏐 LNV Scraper - {total} matchs détectés au total (était 0 avant)")
df_summary

🏐 LNV Scraper - 369 matchs détectés au total (était 0 avant)


,Code,Nom,Matchs,Connue
0,FAZ,FAZ SAFORELLE POWER 6 - PLAYOFFS,0,✅
1,SPS,SPS SAFORELLE POWER 6,124,✅
2,MSL,MSL MARMARA SPIKELIGUE,147,✅
3,PAZ,PAZ MARMARA SPIKELIGUE - PLAYOFFS,0,✅
4,DAZ,DAZ LIGUE A MASCULINE - QUALIFICATION EUROPE,0,✅
5,LBM,LBM LIGUE B MASCULINE,98,✅
6,PBA,PBA LIGUE B MASCULINE - PLAYOFFS - POULE A,0,✅
7,PBB,PBB LIGUE B MASCULINE - PLAYOFFS - POULE B,0,✅
8,PBZ,PBZ LIGUE B MASCULINE - PLAYOFFS,0,✅
9,TSA,TSA TEST POULE FDME A,erreur,🆕
